In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#import pytorch_lightning as pl
import torch
import anndata as ad
from geome import iterables, transforms, ann2data
from geome.ann2data.by_category import Ann2DataByCategory
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import scanpy as sc
from graph_transformer_long_range_niches.pp.geome_utils import prepare_geome_dataset, split_adata
from graph_transformer_long_range_niches.config import load_config
from sklearn.model_selection import train_test_split
from graph_transformer_long_range_niches._paths import CFG_FILES, HE22_HUMAN_LUNG_DATA_PATH
from pathlib import Path

In [3]:
cfg_path = Path(CFG_FILES, 'pancreas_gnntrans_ct.yaml')
cfg = load_config(cfg_path)

# Train, val, test data

In [4]:
SPATIAL_PANCREAS = '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/jimenz_spatial_pancreas.h5ad'
adata = sc.read_h5ad(SPATIAL_PANCREAS)
adata

AnnData object with n_obs × n_vars = 108711 × 979
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.GCG', 'Max.GCG', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'condition', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_NegPrb', 'log1p_total_counts_NegPrb', 'pct_counts_NegPrb', 'n_genes', 'cell_type_coarse', 'x', 'y', 'sliding_window'
    var: 'NegPrb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'log1p', 'spatial'
    obsm: 'spatial', 'spatial_fov'
    layers: 'counts'

In [9]:
if 'classification' in 'node_classification':
    print('YES')

YES


In [5]:
adata = split_adata(adata, split_obs = 'fov', val_size = 0.1, test_size = 0.1, seed = 42)

{'counts': {'train': 85186, 'test': 14636, 'val': 8889}, 'groups': {'train': ['10', '14', '2', '22', '6', '3', '13', '16', '4', '5', '23', '18', '21', '24', '8', '11', '15', '20', '7'], 'val': ['9', '19'], 'test': ['17', '12', '1']}}


0          test
1          test
2          test
3          test
4          test
          ...  
108706    train
108707    train
108708    train
108709    train
108710    train
Name: split, Length: 108711, dtype: object

Can I use scanpy subsample for train, test splitting? https://github.com/scverse/scanpy/blob/c6766d758b83410e9167578d22054f712d5bca4b/src/scanpy/preprocessing/_simple.py#L778

https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.subsample.html

In [ ]:
sc.pp.subsample(data, fraction=None, n_obs=None, random_state=0, copy=False)

# NEW: Test adata_split

In [5]:
from graph_transformer_long_range_niches.pp.geome_utils import prepare_geome_dataset, split_adata

In [6]:
# Is there train, test and val in the data object?

In [8]:
adata_split = split_adata(adata, 'fov', 0.1, 0.1, seed = 42)

{'counts': {'train': 85186, 'test': 14636, 'val': 8889}, 'groups': {'train': ['10', '14', '2', '22', '6', '3', '13', '16', '4', '5', '23', '18', '21', '24', '8', '11', '15', '20', '7'], 'val': ['9', '19'], 'test': ['17', '12', '1']}}


In [11]:
np.unique(adata_split.obs.split)

array(['test', 'train', 'val'], dtype=object)

## Test datasplitting

In [3]:
HE22_HUMAN_LUNG_DATA_PATH = '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/he22_cosmx_human_lung.h5ad'
PANCREAS_CFG = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/example.yaml'

In [19]:
if cfg.dataset.subset_dict == {}:
    print('True')
if not cfg.dataset.subset_dict == {}:
    print('False')

True


In [6]:
cfg.dataset.merge_from_list(['num_features', 3])

In [7]:
cfg.dataset

CfgNode({'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/jimenz_spatial_pancreas.h5ad', 'name': 'pancreas', 'description': "Spatial pancreas data with graphs for each .obs['sliding_window'] and prediction node .obs['niche'].", 'prediction_task': 'node', 'prediction_obs': 'cell_type_coarse', 'library_key': 'sliding_window', 'fine_tuning': [], 'subset_dict': CfgNode({}), 'spatial_neigbors_kwargs': CfgNode({'radius': 50, 'coord_type': 'generic', 'library_key': 'sliding_window'}), 'batch_size': 20, 'train_size': 0.8, 'val_size': 0.2, 'test_size': 0.0, 'num_features': 3, 'num_classes': -1})

In [25]:
spatial_neigbors_kwargs = cfg.dataset.spatial_neigbors_kwargs
spatial_neigbors_kwargs['library_key'] = 'fov'
for k, v in spatial_neigbors_kwargs.items():
    print(k)
    print(v)

radius
50
coord_type
generic
library_key
fov


In [5]:
cfg = load_config(PANCREAS_CFG)

# Geome dataloader
print('Load PyG data...')
pyg_datas, adata, cfg = prepare_geome_dataset(cfg)

Load PyG data...
call new


/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:38: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")


{'cell_type_coarse': ['Acinar', 'Alpha', 'Beta', 'Ductal', 'Endocrine', 'Endothelial', 'Fibroblasts', 'Immune', 'Mast']}
[Data(x=[814, 979], edge_index=[2, 826], y=[814, 9], obs_names=[814]), Data(x=[727, 979], edge_index=[2, 696], y=[727, 9], obs_names=[727]), Data(x=[918, 979], edge_index=[2, 940], y=[918, 9], obs_names=[918])]
144


Follow guidelines from [geome notebooks](https://github.com/theislab/geome/blob/main/docs/notebooks/1_iterables_and_iterators.ipynb).
[Contributing guide](https://scanpy.readthedocs.io/en/latest/dev/getting-set-up.html).

In [4]:
adata = sc.read_h5ad(HE22_HUMAN_LUNG_DATA_PATH)
adata

/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 771203 × 960
    obs: 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'niche', 'image_id', 'cell_ID', 'sex_ontology_term_id', 'assay_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'condition_id', 'donor_id', 'author_cell_type', 'library_key', 'assay', 'organism', 'sex', 'tissue', 'dataset', 'x', 'y', '_scvi_batch', '_scvi_labels', 'window', 'sliding_window'
    var: 'level_0', 'index', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'adj_matrix_neighbors', 'log1p', 'neighbors', 'niche', 'niche_colors', 'schema_version', 'title', 'umap'
    obsm: 'X_pca', 'X_scvi', 'spatial'
    layers: 'log1p', 'raw'

In [5]:
GRAPH_ID = 'sliding_window' # obs variable to split adata on (create graphs from)

In [6]:
adata.obs[GRAPH_ID].cat.categories

Index(['0_0_0', '0_0_1', '0_0_2', '0_0_3', '0_0_4', '0_0_5', '0_1_0', '0_1_1',
       '0_1_2', '0_1_3',
       ...
       '7_4_0', '7_4_1', '7_4_2', '7_4_3', '7_4_4', '7_5_0', '7_5_1', '7_5_2',
       '7_5_3', '7_5_4'],
      dtype='object', length=333)

In [7]:
adata.obs["donor_id"].cat.categories

Index(['Lung5', 'Lung6', 'Lung9', 'Lung12', 'Lung13'], dtype='object')

In [5]:
from graph_transformer_long_range_niches.pp.geome_utils import prepare_geome_dataset
from graph_transformer_long_range_niches.tl.load_config import Config

cfg_path = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/He23/he23_gnntrans_niche.yaml'
cfg = Config(cfg_path)
pyg_datas, adata = prepare_geome_dataset(cfg)


Load cfg file...
{'out_dir': 'results', 'wandb': {'use': True, 'project_name': 'GTLongRange', 'run_idx': None}, 'model': {'model_type': 'gnn-transformer', 'n_epochs': 100}, 'optim': {'lr': 0.001, 'wd': '1e-3', 'warm_up': 10, 'seed': 42}, 'dataset': {'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/he22_cosmx_human_lung.h5ad', 'name': 'he23', 'description': "CosmX Lung data from He23 with graphs for each .obs['sliding_window'] and prediction node .obs['niche'].", 'prediction_task': 'node', 'prediction_obs': 'niche', 'library_key': 'sliding_window', 'subset_dict': {}, 'spatial_neigbors_kwargs': {'radius': 50, 'coord_type': 'generic'}, 'batch_size': 20, 'train_size': 0.8, 'val_size': 0.2, 'test_size': 0.0}, 'gnn': {'gnn_type': 'GCN', 'num_layers': 2, 'hidden_dim': 256, 'embed_dim': 256, 'dropout': 0.1}, 'transformer': {'d_model': 128, 'n_heads': 4, 'dim_feedforward': 512, 'dropout': 0.3, 'num_layers': 4, 'activation_func': 'relu', 'n

/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


call new


/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:38: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")


{'niche': ['immune', 'lymphoid structure', 'macrophages', 'myeloid-enriched stroma', 'neutrophils', 'plasmablast-enriched stroma', 'stroma', 'tumor interior', 'tumor-stroma boundary']}
[Data(x=[1932, 960], edge_index=[2, 1584], y=[1932, 9], obs_names=[1932]), Data(x=[4250, 960], edge_index=[2, 6540], y=[4250, 9], obs_names=[4250]), Data(x=[3427, 960], edge_index=[2, 4182], y=[3427, 9], obs_names=[3427])]
333


### Preprocess

Prepare data format: 

1. calculates addjacency matrix with spatial neighbours
2. select specific data points (?)

In [24]:
from graph_transformer_long_range_niches.tl.load_config import Config  # noqa, register custom modules

cfg_path = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/He23/he23_gnntrans_niche.yaml'
cfg = Config(cfg_path)

Load cfg file...
{'out_dir': 'results', 'wandb': {'use': True, 'project_name': 'GTLongRange', 'run_idx': None}, 'model': {'model_type': 'gnn-transformer', 'n_epochs': 100}, 'optim': {'lr': 0.001, 'wd': '1e-3', 'warm_up': 10, 'seed': 42}, 'dataset': {'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/he22_cosmx_human_lung.h5ad', 'name': 'he23', 'description': "CosmX Lung data from He23 with graphs for each .obs['sliding_window'] and prediction node .obs['niche'].", 'prediction_task': 'node', 'prediction_obs': 'niche', 'library_key': 'sliding_window', 'subset_dict': {}, 'spatial_neigbors_kwargs': {'radius': 50, 'coord_type': 'generic'}, 'batch_size': 20, 'train_size': 0.8, 'val_size': 0.2, 'test_size': 0.0}, 'gnn': {'gnn_type': 'GCN', 'num_layers': 2, 'hidden_dim': 256, 'embed_dim': 256, 'dropout': 0.1}, 'transformer': {'d_model': 128, 'n_heads': 4, 'dim_feedforward': 512, 'dropout': 0.3, 'num_layers': 4, 'activation_func': 'relu', 'n

In [25]:
adj_matrix_loc = "adj_matrix"
fields = {
    "x": ["X"],
    "y": [f"obs/{cfg.get('dataset/prediction_obs')}"],
    "edge_index": ["uns/edge_index"],
}
category_to_iterate = str(cfg.get('dataset/library_key'))
subset_dict = cfg.get('dataset/subset_dict')
spatial_neigbors_kwargs = cfg.get('dataset/spatial_neigbors_kwargs')
spatial_neigbors_kwargs['library_key'] = category_to_iterate

In [19]:
preprocess = transforms.Compose(
        [
            transforms.Subset(key_value = subset_dict, axis="obs"), 
            #transforms.Categorize(keys=list(subset_dict.keys()) + [cfg.get('dataset/prediction_obs'), cfg.get('dataset/library_key')], axis="obs"),
            transforms.Categorize(keys=[cfg.get('dataset/prediction_obs')], axis="obs"),
        ]
    )

In [20]:
d = {"donor_id": ["Lung5"]}
for k, v in d.items():
    print(k)

donor_id


### Transform

In [21]:
transform = transforms.Compose(
    [
        transforms.AddEdgeIndex(edge_index_key="edge_index", func_args=spatial_neigbors_kwargs, spatial_key="spatial", key_added=adj_matrix_loc),
    ]
)

## Load from geome

In [ ]:
from graph_transformer_long_range_niches.tl.load_config import Config  # noqa, register custom modules
from graph_transformer_long_range_niches.pp.geome_utils import prepare_geome_dataset
from graph_transformer_long_range_niches.pp.datamodule_geome import GraphAnnDataModule

In [26]:
cfg_path = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/He23/he23_gnntrans_niche.yaml'

In [27]:
cfg = Config(cfg_path)

Load cfg file...
{'out_dir': 'results', 'wandb': {'use': True, 'project_name': 'GTLongRange', 'run_idx': None}, 'model': {'model_type': 'gnn-transformer', 'n_epochs': 100}, 'optim': {'lr': 0.001, 'wd': '1e-3', 'warm_up': 10, 'seed': 42}, 'dataset': {'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/he22_cosmx_human_lung.h5ad', 'name': 'he23', 'description': "CosmX Lung data from He23 with graphs for each .obs['sliding_window'] and prediction node .obs['niche'].", 'prediction_task': 'node', 'prediction_obs': 'niche', 'library_key': 'sliding_window', 'subset_dict': {}, 'spatial_neigbors_kwargs': {'radius': 50, 'coord_type': 'generic'}, 'batch_size': 20, 'train_size': 0.8, 'val_size': 0.2, 'test_size': 0.0}, 'gnn': {'gnn_type': 'GCN', 'num_layers': 2, 'hidden_dim': 256, 'embed_dim': 256, 'dropout': 0.1}, 'transformer': {'d_model': 128, 'n_heads': 4, 'dim_feedforward': 512, 'dropout': 0.3, 'num_layers': 4, 'activation_func': 'relu', 'n

In [28]:
pyg_datas, adata = prepare_geome_dataset(cfg)

/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:38: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")


{'niche': ['immune', 'lymphoid structure', 'macrophages', 'myeloid-enriched stroma', 'neutrophils', 'plasmablast-enriched stroma', 'stroma', 'tumor interior', 'tumor-stroma boundary']}
[Data(x=[1932, 960], edge_index=[2, 1584], y=[1932, 9], obs_names=[1932]), Data(x=[4250, 960], edge_index=[2, 6540], y=[4250, 9], obs_names=[4250]), Data(x=[3427, 960], edge_index=[2, 4182], y=[3427, 9], obs_names=[3427])]
333


In [30]:
adata

AnnData object with n_obs × n_vars = 771203 × 960
    obs: 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'niche', 'image_id', 'cell_ID', 'sex_ontology_term_id', 'assay_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'condition_id', 'donor_id', 'author_cell_type', 'library_key', 'assay', 'organism', 'sex', 'tissue', 'dataset', 'x', 'y', '_scvi_batch', '_scvi_labels', 'window', 'sliding_window'
    var: 'level_0', 'index', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'adj_matrix_neighbors', 'log1p', 'neighbors', 'niche', 'niche_colors', 'schema_version', 'title', 'umap'
    obsm: 'X_pca', 'X_scvi', 'spatial'
    layers: 'log1p', 'raw'